# Fine-tuning ProtiCelli and building a weights file

This notebook shows where a team's own model work fits in a CM4AI DREAM Challenge submission. A submission is three things: a WDL workflow, a Docker image, and **one weights file**. This notebook produces that weights file:

1. Download the published ProtiCelli weights and the reference data the baseline uses to recognize proteins (a BLAST database and a gene map built from UniProt).
2. Prepare training images from the challenge sample set.
3. Fine-tune ProtiCelli with its own `Model.fit()`, watching the training loss as it runs.
4. Package the fine-tuned model and the reference data into a new weights zip, ready for `blast_proticelli.wdl`.

The fine-tuning here is deliberately small. The goal is to show the workflow, not to improve the model; see the notes in section 5 before reading anything into the results.

**Runs on** a local machine (CUDA or Apple MPS GPU, or CPU for a very slow test) or in a Terra/AnVIL Jupyter cloud environment with a GPU.

## 1. Settings

`RUN_MODE = "quick"` trains for a few epochs, enough to exercise every step and produce a working weights zip in minutes on a GPU. `"full"` trains for longer. Everything this notebook creates goes under `WORK_DIR`, next to the notebook, so deleting that folder resets the notebook.

In [ ]:
from pathlib import Path

RUN_MODE = "quick"          # "quick" or "full"

# Where the training images come from. Copied into WORK_DIR/data, so the source is never modified.
#   Terra:  a gs:// folder, e.g. "gs://<workspace-bucket>/<folder>"
#   Local:  a folder on this machine
# Leave as None if WORK_DIR/data already holds the images.
DATA_SOURCE = None

WORK_DIR = Path("work").resolve()        # everything this notebook writes
DATA_DIR = WORK_DIR / "data"             # sample images and FASTA files, one set per {id}
DOWNLOAD_DIR = WORK_DIR / "downloads"    # ProtiCelli weights, UniProt files
SEED = 1

# Optional: an existing BLAST-ProtiCelli weights zip (for example blast_proticelli_weights_v2.zip).
# Its ProtiCelli model folders are reused instead of downloading them again (about 6.5 GB).
# The UniProt reference data is always rebuilt, so the new weights file gets the current gene map.
EXISTING_WEIGHTS_ZIP = None

TEST_SIZE = 40              # images held out from training, for comparing old and new weights
MIN_BLAST_IDENTITY = 95.0   # see section 4: when the FASTA header cannot name the protein

# Training. Full fine-tuning of ProtiCelli (about 460M parameters) needs roughly 8 GB of GPU
# memory before images; lower BATCH_SIZE if training runs out of memory.
EPOCHS = {"quick": 3, "full": 50}[RUN_MODE]
BATCH_SIZE = 4
LEARNING_RATE = 1e-5        # peak; ProtiCelli's fit() default is 1e-4, gentler here for a small dataset

# Section 6: diffusion steps per prediction, and how many held-out images to compare on
PREDICTION_STEPS = {"quick": 20, "full": 50}[RUN_MODE]
COMPARE_IMAGES = {"quick": 8, "full": TEST_SIZE}[RUN_MODE]

print(f"RUN_MODE={RUN_MODE}, epochs={EPOCHS}, work dir: {WORK_DIR}")

## 2. Environment

Installs this repository's code, which pulls in upstream ProtiCelli pinned to one commit (see `pyproject.toml`). On Terra this can take several minutes, mostly for PyTorch; run it once per cloud environment.

The notebook expects to sit in the `training/` folder of a clone of the repository:

```bash
git clone https://github.com/cm4ai-anvil-dream-challenge/blast-proticelli.git
```

In [ ]:
import subprocess, sys

REPO_DIR = Path.cwd().parent
assert (REPO_DIR / "pyproject.toml").is_file(), f"Expected the repository root at {REPO_DIR}; open this notebook from blast-proticelli/training/"

try:
    import proticelli  # noqa: F401
    print("ProtiCelli already installed:", proticelli.__file__)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", str(REPO_DIR)], check=True)
    print("Installed. If the next cell fails to import, restart the kernel and run from the top.")

BLAST+ is needed to build the BLAST database and to identify proteins whose FASTA header cannot be resolved. If `blastp` is not already installed, on Linux (including Terra) this cell downloads NCBI's prebuilt BLAST+ 2.17.0, the version used for the existing weights files. On a Mac, install it first with `conda install -c bioconda blast`.

In [ ]:
import os, platform, shutil, tarfile, urllib.request

BLAST_VERSION = "2.17.0"
TOOLS_DIR = WORK_DIR / "tools"

def ensure_blast():
    if shutil.which("blastp") and shutil.which("makeblastdb"):
        return
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("blastp not found. Install BLAST+ (for example: conda install -c bioconda blast) and rerun.")
    name = f"ncbi-blast-{BLAST_VERSION}+-x64-linux.tar.gz"
    url = f"https://ftp.ncbi.nlm.nih.gov/blast/executables/blast+/{BLAST_VERSION}/{name}"
    TOOLS_DIR.mkdir(parents=True, exist_ok=True)
    archive = TOOLS_DIR / name
    if not archive.exists():
        print("Downloading", url)
        urllib.request.urlretrieve(url, archive)
    with tarfile.open(archive) as t:
        t.extractall(TOOLS_DIR)
    os.environ["PATH"] = f"{TOOLS_DIR / f'ncbi-blast-{BLAST_VERSION}+' / 'bin'}{os.pathsep}{os.environ['PATH']}"

ensure_blast()
print(subprocess.run(["blastp", "-version"], capture_output=True, text=True).stdout.splitlines()[0])

### GPU check

Training needs a GPU to finish in reasonable time. On Terra, the GPU is chosen when the cloud environment is created; if this cell reports CPU, recreate the environment with a GPU attached.

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("CUDA GPU:", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB)")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple GPU (MPS)")
else:
    DEVICE = "cpu"
    print("No GPU found: training will be very slow. Fine for checking the steps run, not for real training.")
print("PyTorch", torch.__version__)

## 3. Downloads

Two downloads, both kept under `WORK_DIR/downloads`:

* **ProtiCelli model weights** from the Lundberg lab (`checkpoint/` and `vae/`, about 6.5 GB), unless `EXISTING_WEIGHTS_ZIP` is set or a previous run already fetched them.
* **UniProt reference data**: the reviewed human proteins as a BLAST database, and a gene map from UniProt accessions and Ensembl protein IDs to HGNC symbols. These are built by `build_human_blast_db.sh` and `build_human_gene_map.sh` in this folder, which can also be run on their own. They always rebuild from the current UniProt release, which is recorded in each zip's `PROVENANCE.txt`.

A download that fails stops the cell with the reason; rerunning the cell picks up where it left off.

In [ ]:
import zipfile

sys.path.insert(0, str(REPO_DIR))      # use this repository's code, not an older installed copy
from proticelli.utils.download import checkpoint_ready, vae_ready, download_checkpoints

PROTICELLI_DIR = DOWNLOAD_DIR / "proticelli"     # will hold checkpoint/ and vae/
TRAINING_SCRIPTS = REPO_DIR / "training"

def proticelli_weights_ready(d):
    # fit() starts from checkpoint/unet_ema; inference loads checkpoint/unet. Require both, plus the VAE.
    ema = d / "checkpoint" / "unet_ema"
    return checkpoint_ready(d) and vae_ready(d) and (ema / "config.json").is_file()

if proticelli_weights_ready(PROTICELLI_DIR):
    print("Reusing ProtiCelli weights in", PROTICELLI_DIR)
elif EXISTING_WEIGHTS_ZIP:
    print("Extracting ProtiCelli weights from", EXISTING_WEIGHTS_ZIP)
    with zipfile.ZipFile(Path(EXISTING_WEIGHTS_ZIP).expanduser()) as zf:
        members = [n for n in zf.namelist() if n.startswith(("checkpoint/unet/", "checkpoint/unet_ema/", "vae/"))]
        zf.extractall(PROTICELLI_DIR, members=members)
else:
    download_checkpoints(PROTICELLI_DIR)

assert proticelli_weights_ready(PROTICELLI_DIR), f"ProtiCelli weights in {PROTICELLI_DIR} are incomplete"
print("ProtiCelli weights ready")

In [ ]:
REF_DIR = DOWNLOAD_DIR / "reference"
BLAST_DB_DIR = REF_DIR / "human_blast_db"
GENE_MAP_DIR = REF_DIR / "human_gene_map"

def build_reference(script, out_dir, product):
    if (out_dir / product).is_file():
        print(f"Reusing {out_dir / product}")
        return
    print(f"Running {script} (downloads from UniProt; a few minutes)")
    proc = subprocess.run(["bash", str(TRAINING_SCRIPTS / script), str(out_dir)], capture_output=True, text=True)
    print(proc.stdout[-2000:])
    if proc.returncode != 0:
        raise RuntimeError(f"{script} failed:\n{proc.stderr[-2000:]}")

build_reference("build_human_blast_db.sh", BLAST_DB_DIR, "human_blast_db.zip")
build_reference("build_human_gene_map.sh", GENE_MAP_DIR, "human_gene_map.zip")
for d in (BLAST_DB_DIR, GENE_MAP_DIR):
    print((d / "PROVENANCE.txt").read_text())

## 4. Training data

### Copy the images

The images are copied from `DATA_SOURCE` into `WORK_DIR/data`, so the source is never changed. On the way, file names are brought to the challenge naming, `{id}_microtubules.png` (some earlier copies of the sample set use `_microtubule`).

Each training example needs five files: `{id}.faa` and four single-channel 8-bit images, `{id}_microtubules`, `{id}_nucleus`, `{id}_er` and `{id}_protein` (the real protein image, which the model learns to predict). PNG is the challenge format; TIFF is accepted too. `{id}_mask` files are not used for training.

In [ ]:
import json
import numpy as np
import pandas as pd

NAME_FIXES = {"_microtubule.": "_microtubules."}
CHANNELS = ("microtubules", "nucleus", "er", "protein")
IMAGE_EXTS = (".png", ".tif", ".tiff")

def challenge_name(name):
    for old, new in NAME_FIXES.items():
        name = name.replace(old, new)
    return name

def copy_training_data(source, dest):
    dest.mkdir(parents=True, exist_ok=True)
    if source is None:
        if not any(dest.glob("*.faa")):
            raise FileNotFoundError(f"No data in {dest}: set DATA_SOURCE in section 1")
        print(f"Using the data already in {dest}")
        return
    source = str(source)
    if source.startswith("gs://"):
        staging = WORK_DIR / "_incoming"
        staging.mkdir(parents=True, exist_ok=True)
        subprocess.run(["gsutil", "-m", "-q", "cp", source.rstrip("/") + "/*", str(staging)], check=True)
        files, transfer = list(staging.iterdir()), shutil.move
    else:
        files, transfer = [p for p in Path(source).expanduser().iterdir() if p.is_file()], shutil.copy2
    copied = renamed = 0
    for p in files:
        target = dest / challenge_name(p.name)
        if target.exists():
            continue
        transfer(str(p), str(target))
        copied += 1
        renamed += target.name != p.name
    print(f"Copied {copied} files into {dest} ({renamed} renamed to the challenge naming)")

copy_training_data(DATA_SOURCE, DATA_DIR)

def find_examples(data_dir):
    examples, incomplete = [], []
    for faa in sorted(data_dir.glob("*.faa")):
        ex_id = faa.stem
        found = {}
        for ch in CHANNELS:
            hits = [data_dir / f"{ex_id}_{ch}{ext}" for ext in IMAGE_EXTS if (data_dir / f"{ex_id}_{ch}{ext}").is_file()]
            if hits:
                found[ch] = hits[0]
        missing = [ch for ch in CHANNELS if ch not in found]
        if missing:
            incomplete.append((ex_id, missing))
        else:
            examples.append({"id": ex_id, "fasta": faa, **{f"{ch}_image": found[ch] for ch in CHANNELS}})
    return examples, incomplete

examples, incomplete = find_examples(DATA_DIR)
print(f"{len(examples)} complete examples")
for ex_id, missing in incomplete:
    print(f"  left out {ex_id}: no {', '.join(missing)} image")

### Name each protein

ProtiCelli learns one embedding per protein in its vocabulary, so `fit()` needs a vocabulary name for every training image. Each FASTA header gives the antibody's Ensembl protein ID, which the gene map turns into an HGNC symbol. When it cannot (the header's Ensembl release can list IDs the current UniProt release no longer does), the sequence is searched with BLAST, and the best hit is used if it is at least `MIN_BLAST_IDENTITY` percent identical.

An image is left out of training, and goes to the held-out set, when neither route gives a protein ProtiCelli knows. Adding new proteins to the vocabulary means enlarging the model's protein embedding table and training the new rows: one of the places a team could take this further.

In [ ]:
import pickle
from importlib import resources
import protein_predictor as pp

pp.LOCAL_GENE_MAP = str(GENE_MAP_DIR / "human_gene_map.tsv")
pp._LOCAL_ENSP_INDEX = None
gene_map = pd.read_csv(pp.LOCAL_GENE_MAP, sep="\t", dtype=str).fillna("")
accession_symbol = dict(zip(gene_map["accession"], gene_map["hgnc_symbol"]))
BLAST_DB_PREFIX = str(BLAST_DB_DIR / "human_reviewed_swissprot")

with open(resources.files("proticelli") / "data" / "antibody_map.pkl", "rb") as f:
    vocabulary = pp.Vocabulary(pickle.load(f).keys())
print(f"ProtiCelli vocabulary: {len(vocabulary.keys):,} proteins")

def label_example(ex):
    fields = pp.parse_header_fields(pp.read_fasta_header(ex["fasta"]))
    header = pp.header_gene_lookup(fields.get("ensembl_protein_id", ""))
    row = {"hpa_antibody_id": fields.get("hpa_antibody_id", ""),
           "ensembl_protein_id": fields.get("ensembl_protein_id", ""),
           "fasta_hgnc_symbol": header["hgnc_symbol"] or "", "fasta_hgnc_source": header["source"],
           "blast_hgnc_symbol": "", "blast_percent_identity": np.nan,
           "protein_label": "", "label_source": "", "excluded_reason": ""}
    if header["source"] == pp.HEADER_GENE_RESOLVED:
        key = vocabulary.resolve(header["hgnc_symbol"])
        if key:
            return {**row, "protein_label": key, "label_source": "fasta_header"}
        return {**row, "excluded_reason": f"{header['hgnc_symbol']} (from the header) is not in ProtiCelli's vocabulary"}
    hits = pp.local_blast_hits(pp.read_fasta(ex["fasta"]), BLAST_DB_PREFIX, 25)
    if not hits:
        return {**row, "excluded_reason": "header unresolved and no BLAST hit"}
    best = hits[0]
    symbol = accession_symbol.get(best["uniprot_accession"], "")
    row.update(blast_hgnc_symbol=symbol, blast_percent_identity=best["percent_identity"])
    key = vocabulary.resolve(symbol)
    if best["percent_identity"] < MIN_BLAST_IDENTITY:
        return {**row, "excluded_reason": f"header unresolved; best BLAST hit {symbol} only {best['percent_identity']}% identical"}
    if not key:
        return {**row, "excluded_reason": f"header unresolved; best BLAST hit {symbol} is not in ProtiCelli's vocabulary"}
    return {**row, "protein_label": key, "label_source": "blast_best_hit"}

table = pd.DataFrame([{"id": ex["id"], **label_example(ex)} for ex in examples])
print(table["label_source"].replace("", "excluded").value_counts().to_string())
table.loc[table["excluded_reason"] != "", ["id", "hpa_antibody_id", "excluded_reason"]]

### Split into training and held-out images

`TEST_SIZE` images are held out, never seen in training, so the fine-tuned weights can be compared with the original ones on images neither has trained on (section 6). Images that could not be labeled always go to the held-out set. The rest are drawn at random, balanced across how each protein was named and how bright its real image is, so dim proteins and BLAST-named proteins appear on both sides.

In the 2026-10 sample set every image shows a different antibody, so the held-out images are also held-out proteins.

In [ ]:
from image_scaling import read_png

table["protein_max"] = [int(read_png(ex["protein_image"]).max()) for ex in examples]
table["brightness"] = np.where(table["protein_max"] < 200, "dim", "bright")
table["stratum"] = table["label_source"].replace("", "excluded") + "/" + table["brightness"]

rng = np.random.default_rng(SEED)
excluded = table.index[table["excluded_reason"] != ""]
labeled = table.drop(excluded)
n_draw = max(TEST_SIZE - len(excluded), 0)
# Proportional allocation across strata, rounding by largest remainder
sizes = labeled["stratum"].value_counts()
quota = sizes / sizes.sum() * n_draw
alloc = np.floor(quota).astype(int)
alloc[(quota - alloc).sort_values(ascending=False).index[: n_draw - alloc.sum()]] += 1
held = list(excluded)
for stratum, k in alloc.items():
    held += list(rng.choice(labeled.index[labeled["stratum"] == stratum], size=k, replace=False))
table["split"] = np.where(table.index.isin(held), "test", "train")

WORK_DIR.mkdir(parents=True, exist_ok=True)
table.to_csv(WORK_DIR / "training_examples.csv", index=False)
print(pd.crosstab(table["stratum"], table["split"]).to_string())
print(f"\nWritten to {WORK_DIR / 'training_examples.csv'}")

### Build the training images

`fit()` reads 4-channel TIFFs. Each is assembled and normalized with ProtiCelli's own `ChannelAssembler` and `ImageNormalizer`, the same steps the baseline applies to its inputs at prediction time, so training and prediction see identically scaled images. The channel order is ProtiCelli's: microtubules, protein, nucleus, ER. (The `fit()` docstring names the channels differently; the code uses positions 0, 2 and 3 as the reference and 1 as the protein, which matches this order.)

The held-out images are not converted: section 6 runs them through the normal prediction pipeline from the original files.

In [ ]:
import tifffile
from proticelli.data import ChannelAssembler, ImageNormalizer

TRAIN_DIR = WORK_DIR / "train_tiff"
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
assembler, normalizer = ChannelAssembler(has_protein=True), ImageNormalizer(bit_depth=8)
by_id = {ex["id"]: ex for ex in examples}

train = table[table["split"] == "train"]
for ex_id in train["id"]:
    out = TRAIN_DIR / f"{ex_id}.tiff"
    if out.exists():
        continue
    ex = by_id[ex_id]
    raw = assembler.transform({ch: read_png(ex[f"{ch}_image"]) for ch in CHANNELS})
    stack = normalizer.transform(raw).astype(np.float32)
    # fit() rescales anything above 1.0 as if it were raw counts; normalized images never are
    assert stack.shape == (512, 512, 4) and stack.min() >= -1 and stack.max() <= 1, ex_id
    tifffile.imwrite(out, stack)

TRAIN_FILES = [f"{i}.tiff" for i in train["id"]]
TRAIN_PROTEINS = list(train["protein_label"])
print(f"{len(TRAIN_FILES)} training images in {TRAIN_DIR}")

In [ ]:
import matplotlib.pyplot as plt

ex = by_id[train["id"].iloc[0]]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, ch in zip(axes, CHANNELS):
    ax.imshow(read_png(ex[f"{ch}_image"]), cmap="gray")
    ax.set_title(ch if ch != "protein" else f"protein: {train['protein_label'].iloc[0]}")
    ax.axis("off")
fig.suptitle(f"Training example {ex['id']}: three reference channels in, protein image to predict")
plt.show()

## 5. Fine-tune

`Model.fit()` trains ProtiCelli on the training images, starting from the published weights. Every step, a batch of real protein images is corrupted with noise of a random strength, and the model learns to recover them given the three reference channels and the protein's name. The plot below updates as training runs: each point is one step's loss, and the line is the average over the last epoch.

**Reading the plot.** The per-step loss jumps around because each step draws a different noise strength; the running average is the thing to watch. A falling average means the model is fitting the training images, not that its predictions for new images got better; section 6 compares the original and fine-tuned weights on held-out images for that. In `"quick"` mode, expect a short, noisy curve.

**Weights.** ProtiCelli ships two copies of its weights: `checkpoint/unet`, which prediction loads, and `checkpoint/unet_ema`, a smoothed running average from its original training. `fit()` starts from `unet_ema` and, as configured here (EMA averaging off), saves the fine-tuned weights to a folder named `unet_ema`. Section 6 packages them as `checkpoint/unet` so prediction uses them. Whether to keep an EMA copy during fine-tuning is a choice teams can revisit.

**What to expect.** The sample set is small and was not designed for training. Whether these images were part of ProtiCelli's original training data is not yet confirmed; neither of the challenge cell lines (MDA-MB-468, KOLF2.1J) is among the 41 cell lines ProtiCelli was trained with, so training runs without a cell line label.

In [ ]:
import math
from IPython.display import display
from proticelli import Model
import proticelli._training as proticelli_training

FINETUNE_DIR = WORK_DIR / "finetune"
steps_per_epoch = math.ceil(len(TRAIN_FILES) / BATCH_SIZE)
total_steps = EPOCHS * steps_per_epoch
warmup_steps = max(1, total_steps // 10)   # fit()'s default of 500 would outlast a short run
print(f"{len(TRAIN_FILES)} images, batch size {BATCH_SIZE}: {steps_per_epoch} steps per epoch, "
      f"{total_steps} steps, learning rate warmup {warmup_steps} steps")

model = Model(checkpoint_dir=PROTICELLI_DIR / "checkpoint", vae_dir=PROTICELLI_DIR / "vae", device=DEVICE)
torch.manual_seed(SEED)

# Live loss plot. fit() reports each step's loss to its progress bar; a progress bar
# that also records the loss lets the plot follow training without changing fit().
step_losses = []
fig, ax = plt.subplots(figsize=(9, 3.5))
plot_handle = display(fig, display_id=True)
plt.close(fig)

def redraw():
    ax.clear()
    ax.plot(range(1, len(step_losses) + 1), step_losses, ".", color="0.7", markersize=3, label="loss per step")
    if len(step_losses) >= 2:
        window = min(steps_per_epoch, len(step_losses))
        smooth = np.convolve(step_losses, np.ones(window) / window, mode="valid")
        ax.plot(range(window, len(step_losses) + 1), smooth, color="C0", label=f"average over {window} steps")
    for epoch in range(1, len(step_losses) // steps_per_epoch + 1):
        ax.axvline(epoch * steps_per_epoch + 0.5, color="0.9", zorder=0)
    ax.set_xlabel("training step (gray lines: epoch boundaries)")
    ax.set_ylabel("loss")
    ax.set_xlim(0, total_steps + 1)
    ax.legend(loc="upper right")
    plot_handle.update(fig)

class LossRecordingProgressBar(proticelli_training.tqdm):
    def set_postfix(self, *args, **kwargs):
        if "loss" in kwargs:
            step_losses.append(float(kwargs["loss"]))
            if len(step_losses) % max(1, steps_per_epoch // 5) == 0 or len(step_losses) == total_steps:
                redraw()
        return super().set_postfix(*args, **kwargs)

original_progress_bar = proticelli_training.tqdm
proticelli_training.tqdm = LossRecordingProgressBar
try:
    model.fit(
        image_dir=TRAIN_DIR,
        image_files=TRAIN_FILES,
        protein_names=TRAIN_PROTEINS,
        output_dir=str(FINETUNE_DIR),
        num_epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        lr_warmup_steps=warmup_steps,
        save_model_epochs=10,
        checkpointing_steps=10**9,                    # no intermediate checkpoints (1.8 GB each)
        num_workers=4 if DEVICE == "cuda" else 0,     # worker processes are unreliable in notebooks on macOS
    )
finally:
    proticelli_training.tqdm = original_progress_bar
redraw()

pd.Series(step_losses, name="loss").rename_axis("step").to_csv(WORK_DIR / "training_loss.csv")
assert (FINETUNE_DIR / "unet_ema" / "config.json").is_file(), "fit() did not save weights"
print("Fine-tuned weights saved in", FINETUNE_DIR / "unet_ema")
del model
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## 6. Package the weights file

The new weights file has the same layout as the baseline's, so `blast_proticelli.wdl` and `run_inference.py` use it unchanged:

```
checkpoint/unet/                fine-tuned ProtiCelli weights
vae/                            ProtiCelli VAE (not trained here, copied as is)
reference/human_blast_db.zip    from section 3
reference/human_gene_map.zip    from section 3
```

It is built by `build_weights_zip.sh`, the same script used for the baseline's own weights files.

In [ ]:
import datetime

PACKAGE_DIR = WORK_DIR / "package"
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
shutil.copytree(FINETUNE_DIR / "unet_ema", PACKAGE_DIR / "checkpoint" / "unet")
shutil.copytree(PROTICELLI_DIR / "vae", PACKAGE_DIR / "vae")

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
NEW_WEIGHTS = WORK_DIR / f"blast_proticelli_weights_finetuned_{RUN_MODE}_{stamp}.zip"
proc = subprocess.run(
    ["bash", str(TRAINING_SCRIPTS / "build_weights_zip.sh"), str(PACKAGE_DIR),
     str(BLAST_DB_DIR / "human_blast_db.zip"), str(GENE_MAP_DIR / "human_gene_map.zip"), str(NEW_WEIGHTS)],
    capture_output=True, text=True)
print(proc.stdout, proc.stderr)
proc.check_returncode()
shutil.rmtree(PACKAGE_DIR)
print(f"New weights file: {NEW_WEIGHTS} ({NEW_WEIGHTS.stat().st_size / 2**30:.1f} GiB)")

### Predict with it

Runs the baseline's own prediction script, `run_inference.py`, on `COMPARE_IMAGES` held-out images, once with the new weights and once with the original ProtiCelli weights, using the same random seed. This is exactly what the workflow does with a submitted weights file. It needs the original weights as a weights zip: `EXISTING_WEIGHTS_ZIP` if set, otherwise one is built from the downloaded ProtiCelli weights.

In [ ]:
held_out = table[table["split"] == "test"].head(COMPARE_IMAGES)
records = [{"prediction_id": r.id,
            "microtubules_image": str(by_id[r.id]["microtubules_image"]),
            "nucleus_image": str(by_id[r.id]["nucleus_image"]),
            "er_image": str(by_id[r.id]["er_image"]),
            "fasta_file": str(by_id[r.id]["fasta"])} for r in held_out.itertuples()]
inputs_json = WORK_DIR / "held_out_inputs.json"
inputs_json.write_text(json.dumps(records, indent=2))

ORIGINAL_WEIGHTS = Path(EXISTING_WEIGHTS_ZIP).expanduser() if EXISTING_WEIGHTS_ZIP else WORK_DIR / "blast_proticelli_weights_original.zip"
if not ORIGINAL_WEIGHTS.exists():
    subprocess.run(["bash", str(TRAINING_SCRIPTS / "build_weights_zip.sh"), str(PROTICELLI_DIR),
                    str(BLAST_DB_DIR / "human_blast_db.zip"), str(GENE_MAP_DIR / "human_gene_map.zip"),
                    str(ORIGINAL_WEIGHTS)], check=True, capture_output=True)

def predict(weights_zip, name):
    out = WORK_DIR / "predictions" / name
    cmd = [sys.executable, str(REPO_DIR / "run_inference.py"), "--inputs", str(inputs_json),
           "--weights_zip", str(weights_zip), "--output_dir", str(out),
           "--weights_dir", str(WORK_DIR / "inference_weights" / name),
           "--reference_data_dir", str(WORK_DIR / "inference_reference" / name),
           "--num_inference_steps", str(PREDICTION_STEPS), "--seed", str(SEED)]
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd=WORK_DIR)
    if proc.returncode != 0:
        raise RuntimeError(proc.stdout[-3000:] + proc.stderr[-3000:])
    return out

pred_dirs = {"original": predict(ORIGINAL_WEIGHTS, "original"), "fine-tuned": predict(NEW_WEIGHTS, "finetuned")}
print("Predictions in", WORK_DIR / "predictions")

### Compare original and fine-tuned weights

A simple, illustrative comparison on the held-out images: the Pearson correlation between each prediction and the real protein image, inside the cell mask when there is one. The challenge's own metrics are still being settled and are computed by the evaluation tools, not here. With `"quick"` settings, differences between the two columns mostly reflect noise.

In [ ]:
def pearson(a, b):
    a, b = a.astype(np.float64).ravel(), b.astype(np.float64).ravel()
    if a.std() == 0 or b.std() == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def mask_for(ex_id):
    for ext in IMAGE_EXTS:
        p = DATA_DIR / f"{ex_id}_mask{ext}"
        if p.is_file():
            return read_png(p) > 0
    return None

rows = []
for r in held_out.itertuples():
    truth = read_png(by_id[r.id]["protein_image"])
    mask = mask_for(r.id)
    row = {"id": r.id, "protein": r.fasta_hgnc_symbol or r.blast_hgnc_symbol, "label_source": r.label_source or "excluded"}
    for name, d in pred_dirs.items():
        pred = read_png(d / f"{r.id}.png")
        row[f"pearson_{name}"] = pearson(pred[mask], truth[mask]) if mask is not None else pearson(pred, truth)
    rows.append(row)
comparison = pd.DataFrame(rows)
comparison["change"] = comparison["pearson_fine-tuned"] - comparison["pearson_original"]
comparison.to_csv(WORK_DIR / "held_out_comparison.csv", index=False)
print(comparison[["pearson_original", "pearson_fine-tuned", "change"]].describe().loc[["mean", "50%", "min", "max"]].round(3).to_string())
comparison.round(3)

In [ ]:
show = held_out["id"].head(4).tolist()
fig, axes = plt.subplots(len(show), 4, figsize=(12, 3 * len(show)), squeeze=False)
for row_axes, ex_id in zip(axes, show):
    panels = [("nucleus (reference)", read_png(by_id[ex_id]["nucleus_image"])),
              ("real protein", read_png(by_id[ex_id]["protein_image"])),
              ("original weights", read_png(pred_dirs["original"] / f"{ex_id}.png")),
              ("fine-tuned weights", read_png(pred_dirs["fine-tuned"] / f"{ex_id}.png"))]
    for ax, (title, img) in zip(row_axes, panels):
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"{ex_id}: {title}", fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

## Next steps

* **Use the weights file in a submission.** Upload `NEW_WEIGHTS` to the workspace bucket and pass it as `weights_file` to `blast_proticelli.wdl`; nothing else in the workflow changes.
* **Where a team could go further:** train for longer or on more data; tune normalization or augmentation in section 4; add proteins to ProtiCelli's vocabulary (section 4); replace the BLAST step with a model that reads the amino acid sequence directly; or swap ProtiCelli for another model entirely, keeping the same inputs and outputs.
* **Housekeeping.** `WORK_DIR` holds several GB (weights copies, training images); delete it when done.